### Objective:

- Save and retrieve processed data efficiently inside Dataproc.
- Serve data in a structured way for analysis.
- Use Parquet, Hive, and CSV 

In [1]:
# Creating the Spark session
from pyspark.sql import SparkSession
import spark

spark = SparkSession.builder \
        .appName("OlistData") \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 17:19:02 INFO SparkEnv: Registering MapOutputTracker
26/04/22 17:19:02 INFO SparkEnv: Registering BlockManagerMaster
26/04/22 17:19:02 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/22 17:19:02 INFO SparkEnv: Registering OutputCommitCoordinator


In [2]:
spark = SparkSession.builder \
.appName('Olist Ecommerce Performance Optmization') \
.config('spark.executor.memory','6g') \
.config('spark.executor.cores','4') \
.config('spark.executor.instances','2') \
.config('spark.driver.memory','4g') \
.config('spark.driver.maxResultSize','2g') \
.config('spark.sql.shuffle.partitions','64') \
.config('spark.default.parallelism','64') \
.config('spark.sql.adaptive.enabled','true') \
.config('spark.sql.adaptive.coalescePartition.enabled','true') \
.config('spark.sql.autoBroadcastJoinThreshold',20*1024*1024) \
.config('spark.sql.files.maxPartitionBytes','64MB') \
.config('spark.sql.files.openCostInBytes','2MB') \
.config('spark.memory.fraction',0.8) \
.config('spark.memory.storageFraction',0.2) \
.getOrCreate()

26/04/22 17:19:17 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
full_orders_df = spark.read.parquet('/tmp/olist/')

In [5]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

#### Storing the data in parquet format in HDFS

In [ ]:
## To CREATE A DIRECTORY IN THE tmp -> olist is name of the folder
!hadoop fs -mkdir /tmp/olist

In [ ]:
### AFTER CREATING THE FOLDER olist -> Writing the data files into the olist
full_orders_df.write.mode('overwrite').parquet('/tmp/olist')

## To check go to cluster Web Interfaces -> HDFS NameNode ->  Utilities -> Browse file system -> tmp -> olist folder

#### Save as a parquet in Google Cloud Storage -> Cloud Storage -> Bucket

In [6]:
full_orders_df.write.mode('overwrite').parquet('gs://dataproc-staging-us-central1-717850913772-zpnet7tg/temp_data')

# To check go to cloud storage -> view bucket list -> stagging bucket -> temp_data folder

#### Save data as a hive table

In [7]:
full_orders_df.write.mode('overwrite').saveAsTable('full_order_detail')

# To check go to cluster Web Interfaces -> JupyterLab -> Terminal -> hive (terminal command) -> show tables (we can find the full_order_detail)

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used
26/04/22 17:32:19 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [8]:
spark.sql('show tables')

DataFrame[namespace: string, tableName: string, isTemporary: boolean]

In [12]:
spark.sql('select * from full_order_detail').show(1)

+--------------------+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-----+-------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+-----------+------------+--------------------+------------------------+-------------+--------------+---------------------------+-------------------+------------------+----------------+-----------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+------------------+------------+--------------------+-------------+------------+-----------+-------------+------------+-----------+----+----------------+-----------+--------------+----------------+
|         

#### Storing the data in csv format in HDFS

In [13]:
full_orders_df.write.mode('append').option('header','true').csv('/tmp/olist/')